In [ ]:
# ============================================================
# D6 — Stage 4 Validation — Branch C: Deterministic Normalisation
# 0. Imports
# ============================================================

from google.colab import files
from pathlib import Path
from datetime import datetime
from difflib import SequenceMatcher
from collections import Counter

import hashlib
import json
import math
import platform
import re
import sys
import unicodedata

import pandas as pd
from scipy.optimize import linear_sum_assignment

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D6"
DOCUMENT_NAME = "Microsoft FY24 Q1 Press Release"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"
INPUT_REPRESENTATION = "Complete deterministically normalised structural Markdown"

EXPECTED_REFERENCE_RECORD_COUNT = 147
EXPECTED_EXTRACTION_RECORD_COUNT = 147

EXPECTED_CATEGORY_COUNTS = {
    "Narrative performance highlight": 24,
    "Financial performance reconciliation": 4,
    "Segment revenue reconciliation": 3,
    "Selected product and service reconciliation": 15,
    "Income statement": 19,
    "Comprehensive income statement": 6,
    "Balance sheet": 34,
    "Cash flow statement": 34,
    "Segment revenue and operating income": 8
}

FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change",
    "Unit",
    "Reporting Period",
    "Source Location"
]

PRIMARY_CORRECTNESS_FIELDS = FIELDS.copy()

STRING_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Unit",
    "Reporting Period",
    "Source Location"
]

NUMERIC_FIELDS = [
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change"
]

# Frozen from final D6 Branch A validation.
BLOCK_FIELDS = [
    "Category",
    "Source Location"
]

MATCH_SCORE_THRESHOLD = 0.34

OUTPUT_DIR = Path("outputs_D6_validation_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Primary correctness fields:", len(PRIMARY_CORRECTNESS_FIELDS))
print("Output directory:", OUTPUT_DIR)

Document: D6
Branch: C
Expected reference records: 147
Primary correctness fields: 12
Output directory: outputs_D6_validation_C


In [ ]:
# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D6_reference_values.csv
#   2) D6_branch_C_combined_parsed_extraction.json
#   3) D6_branch_C_structure_check.json
#   4) D6_branch_C_normalisation_check.json
#
# Optional:
#   5) D6_branch_C_experiment_summary.json

print(
    "Upload:\n"
    "1. D6_reference_values.csv\n"
    "2. D6_branch_C_combined_parsed_extraction.json\n"
    "3. D6_branch_C_structure_check.json\n"
    "4. D6_branch_C_normalisation_check.json\n"
    "5. Optional: D6_branch_C_experiment_summary.json"
)

uploaded = files.upload()

csv_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".csv")
]

json_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".json")
]

if len(csv_paths) != 1:
    raise ValueError("Upload exactly one CSV reference file.")

REFERENCE_PATH = csv_paths[0]

EXTRACTION_PATH = None
STRUCTURE_CHECK_PATH = None
NORMALISATION_CHECK_PATH = None
EXPERIMENT_SUMMARY_PATH = None

for path in json_paths:
    with path.open("r", encoding="utf-8-sig") as file:
        obj = json.load(file)

    if not isinstance(obj, dict):
        continue

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        EXTRACTION_PATH = path

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structure_valid" in obj
        and "record_schema_valid" in obj
        and "field_types_valid" in obj
    ):
        STRUCTURE_CHECK_PATH = path

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and obj.get("parent_branch") == PARENT_BRANCH
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_CHECK_PATH = path

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "validation_status" in obj
        and "accuracy_validation_completed" in obj
    ):
        EXPERIMENT_SUMMARY_PATH = path

if EXTRACTION_PATH is None:
    raise ValueError(
        "Could not identify D6_branch_C_combined_parsed_extraction.json."
    )

if STRUCTURE_CHECK_PATH is None:
    raise ValueError(
        "Could not identify D6_branch_C_structure_check.json."
    )

if NORMALISATION_CHECK_PATH is None:
    raise ValueError(
        "Could not identify D6_branch_C_normalisation_check.json."
    )

print("Reference:", REFERENCE_PATH.name)
print("Extraction:", EXTRACTION_PATH.name)
print("Structure check:", STRUCTURE_CHECK_PATH.name)
print("Normalisation check:", NORMALISATION_CHECK_PATH.name)
print(
    "Experiment summary:",
    EXPERIMENT_SUMMARY_PATH.name
    if EXPERIMENT_SUMMARY_PATH is not None
    else "Not supplied"
)

Upload:
1. D6_reference_values.csv
2. D6_branch_C_combined_parsed_extraction.json
3. D6_branch_C_structure_check.json
4. D6_branch_C_normalisation_check.json
5. Optional: D6_branch_C_experiment_summary.json


Saving D6_branch_C_experiment_summary.json to D6_branch_C_experiment_summary.json
Saving D6_branch_C_structure_check.json to D6_branch_C_structure_check.json
Saving D6_branch_C_combined_parsed_extraction.json to D6_branch_C_combined_parsed_extraction.json
Saving D6_branch_C_normalisation_check.json to D6_branch_C_normalisation_check.json
Saving D6_reference_values.csv to D6_reference_values.csv
Reference: D6_reference_values.csv
Extraction: D6_branch_C_combined_parsed_extraction.json
Structure check: D6_branch_C_structure_check.json
Normalisation check: D6_branch_C_normalisation_check.json
Experiment summary: D6_branch_C_experiment_summary.json


In [ ]:
# ============================================================
# 3. File hashing utility and input provenance
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
STRUCTURE_CHECK_SHA256 = sha256_file(STRUCTURE_CHECK_PATH)
NORMALISATION_CHECK_SHA256 = sha256_file(NORMALISATION_CHECK_PATH)

EXPERIMENT_SUMMARY_SHA256 = (
    sha256_file(EXPERIMENT_SUMMARY_PATH)
    if EXPERIMENT_SUMMARY_PATH is not None
    else None
)

print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print("Structure check SHA-256:", STRUCTURE_CHECK_SHA256)
print("Normalisation check SHA-256:", NORMALISATION_CHECK_SHA256)

Reference SHA-256: 534e075d05608e9a993a4f7ac432187c3cf1e8611be71a9acfbfba9ed35ab5cf
Extraction SHA-256: b24e56afae83effc5ef7c8123f09a9f196bfc0512726ea3a9063ba20a76bb6f3
Structure check SHA-256: 047f2608b7e18ffd3da51cb9df318a2ddb8827b24ae434e55f9130aae0280f38
Normalisation check SHA-256: 0a43c5d30fce92fabe46caeba05edfbb9c5a67f2e705e7a148904c5d6ac13c96


In [ ]:
# ============================================================
# 4. Load fixed Stage 1 reference dataset
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
)


def restore_csv_null(value):
    if value is None:
        return None

    if isinstance(value, str) and value == "":
        return None

    return value


def restore_mixed_number(value):
    if value is None or not isinstance(value, str):
        return value

    text = (
        value.strip()
        .replace(",", "")
        .replace("−", "-")
        .replace("–", "-")
    )

    if re.fullmatch(r"-?\d+", text):
        return int(text)

    if re.fullmatch(r"-?\d+\.\d+", text):
        return float(text)

    return value


for column in reference_df.columns:
    reference_df[column] = reference_df[column].map(restore_csv_null)

for field in NUMERIC_FIELDS:
    reference_df[field] = reference_df[field].map(restore_mixed_number)


print("Reference shape:", reference_df.shape)
print("Reference columns:", reference_df.columns.tolist())

display(reference_df.head(10))

Reference shape: (147, 12)
Reference columns: ['Category', 'Statement or Section', 'Metric', 'Business Area', 'Value 2023', 'Value 2022', 'GAAP YoY Change', 'Constant Currency Impact', 'Constant Currency YoY Change', 'Unit', 'Reporting Period', 'Source Location']


,Category,Statement or Section,Metric,Business Area,Value 2023,Value 2022,GAAP YoY Change,Constant Currency Impact,Constant Currency YoY Change,Unit,Reporting Period,Source Location
0,Narrative performance highlight,Quarterly results and Business Highlights,Revenue,Corporate,56.50,NaN,13.0,NaN,12.0,USD billion,"Quarter ended September 30, 2023",Page 1 — Quarterly results
1,Narrative performance highlight,Quarterly results and Business Highlights,Operating income,Corporate,26.90,NaN,25.0,NaN,24.0,USD billion,"Quarter ended September 30, 2023",Page 1 — Quarterly results
2,Narrative performance highlight,Quarterly results and Business Highlights,Net income,Corporate,22.30,NaN,27.0,NaN,26.0,USD billion,"Quarter ended September 30, 2023",Page 1 — Quarterly results
3,Narrative performance highlight,Quarterly results and Business Highlights,Diluted earnings per share,Corporate,2.99,NaN,27.0,NaN,26.0,USD per share,"Quarter ended September 30, 2023",Page 1 — Quarterly results
4,Narrative performance highlight,Quarterly results and Business Highlights,Microsoft Cloud revenue,Microsoft Cloud,31.80,NaN,24.0,NaN,23.0,USD billion,"Quarter ended September 30, 2023",Page 1 — Quarterly results
5,Narrative performance highlight,Quarterly results and Business Highlights,Revenue,Productivity and Business Processes,18.60,NaN,13.0,NaN,12.0,USD billion,"Quarter ended September 30, 2023",Page 1 — Business Highlights
6,Narrative performance highlight,Quarterly results and Business Highlights,Office Commercial products and cloud services ...,Office Commercial,NaN,NaN,15.0,NaN,14.0,percent,"Quarter ended September 30, 2023",Page 1 — Business Highlights
7,Narrative performance highlight,Quarterly results and Business Highlights,Office 365 Commercial revenue,Office 365 Commercial,NaN,NaN,18.0,NaN,17.0,percent,"Quarter ended September 30, 2023",Page 1 — Business Highlights
8,Narrative performance highlight,Quarterly results and Business Highlights,Office Consumer products and cloud services re...,Office Consumer,NaN,NaN,3.0,NaN,4.0,percent,"Quarter ended September 30, 2023",Page 1 — Business Highlights
9,Narrative performance highlight,Quarterly results and Business Highlights,Microsoft 365 Consumer subscribers,Microsoft 365 Consumer,76.70,NaN,NaN,NaN,NaN,million subscribers,"Quarter ended September 30, 2023",Page 1 — Business Highlights


In [ ]:
# ============================================================
# 5. Load untouched Branch C combined extraction
# ============================================================

with EXTRACTION_PATH.open("r", encoding="utf-8") as file:
    extraction_content = json.load(file)

valid_json = True
top_level_object_valid = isinstance(extraction_content, dict)

document_id_correct = (
    top_level_object_valid
    and extraction_content.get("document_id") == DOCUMENT_ID
)

branch_correct = (
    top_level_object_valid
    and extraction_content.get("branch") == BRANCH
)

records_is_list = (
    top_level_object_valid
    and isinstance(extraction_content.get("records"), list)
)

if not records_is_list:
    raise ValueError(
        "Expected the Branch C combined extraction to be a JSON object "
        "containing document_id, branch and a records list."
    )

extracted_records = extraction_content["records"]

print("Valid JSON:", valid_json)
print("Top-level object valid:", top_level_object_valid)
print("Document ID correct:", document_id_correct)
print("Branch correct:", branch_correct)
print("Records is list:", records_is_list)
print("Extracted records:", len(extracted_records))

Valid JSON: True
Top-level object valid: True
Document ID correct: True
Branch correct: True
Records is list: True
Extracted records: 147


In [ ]:
# ============================================================
# 6. Load Branch C structural diagnostics
# ============================================================

with STRUCTURE_CHECK_PATH.open("r", encoding="utf-8") as file:
    branch_structure_check = json.load(file)

branch_structure_document_id_correct = (
    branch_structure_check.get("document_id") == DOCUMENT_ID
)

branch_structure_branch_correct = (
    branch_structure_check.get("branch") == BRANCH
)

branch_structure_valid = bool(
    branch_structure_check.get("structure_valid", False)
)

print("Structure-check document ID correct:", branch_structure_document_id_correct)
print("Structure-check branch correct:", branch_structure_branch_correct)
print("Branch C structure_valid:", branch_structure_valid)
print(
    "Branch C record_schema_valid:",
    branch_structure_check.get("record_schema_valid")
)
print(
    "Branch C field_types_valid:",
    branch_structure_check.get("field_types_valid")
)
print(
    "Branch C scope_complete:",
    branch_structure_check.get("scope_complete")
)

Structure-check document ID correct: True
Structure-check branch correct: True
Branch C structure_valid: True
Branch C record_schema_valid: True
Branch C field_types_valid: True
Branch C scope_complete: True


In [ ]:
# ============================================================
# 7. Preserve raw record structure and create comparison table
# ============================================================

raw_record_rows = []
schema_issue_rows = []

for record_index, record in enumerate(extracted_records):

    if not isinstance(record, dict):
        schema_issue_rows.append(
            {
                "Record Index": record_index,
                "Issue": "Record is not a JSON object",
                "Missing Fields": FIELDS,
                "Extra Fields": None,
                "Expected Fields": FIELDS,
                "Observed Fields": None
            }
        )

        comparison_record = {field: None for field in FIELDS}

    else:
        observed_fields = list(record.keys())

        missing_fields = [
            field
            for field in FIELDS
            if field not in record
        ]

        extra_fields = [
            field
            for field in observed_fields
            if field not in FIELDS
        ]

        # Field order is recorded diagnostically but is not required
        # for semantic JSON object validity.
        field_order_valid = observed_fields == FIELDS

        if missing_fields or extra_fields:
            schema_issue_rows.append(
                {
                    "Record Index": record_index,
                    "Issue": "Missing or unexpected field names",
                    "Missing Fields": missing_fields,
                    "Extra Fields": extra_fields,
                    "Expected Fields": FIELDS,
                    "Observed Fields": observed_fields,
                    "Field Order Valid": field_order_valid
                }
            )

        # Comparison copy only:
        # absent expected fields become None.
        # Unexpected fields are NOT renamed, repaired or copied.
        comparison_record = {
            field: record.get(field)
            for field in FIELDS
        }

    comparison_record["_extraction_index"] = record_index
    raw_record_rows.append(comparison_record)


extracted_df = pd.DataFrame(raw_record_rows)
schema_issues_df = pd.DataFrame(schema_issue_rows)

record_schema_valid = schema_issues_df.empty

print("Record schema valid:", record_schema_valid)
print("Schema issue count:", len(schema_issues_df))

if not schema_issues_df.empty:
    display(schema_issues_df)

display(extracted_df.head(10))

Record schema valid: True
Schema issue count: 0


,Category,Statement or Section,Metric,Business Area,Value 2023,Value 2022,GAAP YoY Change,Constant Currency Impact,Constant Currency YoY Change,Unit,Reporting Period,Source Location,_extraction_index
0,Narrative performance highlight,Quarterly results,Revenue,None,56.50,NaN,13.0,NaN,12.0,$ billion / %,"Quarter ended September 30, 2023",Page 1 — Quarterly results,0
1,Narrative performance highlight,Quarterly results,Operating income,None,26.90,NaN,25.0,NaN,24.0,$ billion / %,"Quarter ended September 30, 2023",Page 1 — Quarterly results,1
2,Narrative performance highlight,Quarterly results,Net income,None,22.30,NaN,27.0,NaN,26.0,$ billion / %,"Quarter ended September 30, 2023",Page 1 — Quarterly results,2
3,Narrative performance highlight,Quarterly results,Diluted earnings per share,None,2.99,NaN,27.0,NaN,26.0,$ per share / %,"Quarter ended September 30, 2023",Page 1 — Quarterly results,3
4,Narrative performance highlight,Quarterly results,Microsoft Cloud revenue,Microsoft Cloud,31.80,NaN,24.0,NaN,23.0,$ billion / %,"Quarter ended September 30, 2023",Page 1 — Quarterly results,4
5,Narrative performance highlight,Business Highlights,Revenue,Productivity and Business Processes,18.60,NaN,13.0,NaN,12.0,$ billion / %,"Quarter ended September 30, 2023",Page 1 — Business Highlights,5
6,Narrative performance highlight,Business Highlights,Office Commercial products and cloud services ...,Productivity and Business Processes,NaN,NaN,15.0,NaN,14.0,%,"Quarter ended September 30, 2023",Page 1 — Business Highlights,6
7,Narrative performance highlight,Business Highlights,Office 365 Commercial revenue,Productivity and Business Processes,NaN,NaN,18.0,NaN,17.0,%,"Quarter ended September 30, 2023",Page 1 — Business Highlights,7
8,Narrative performance highlight,Business Highlights,Office Consumer products and cloud services re...,Productivity and Business Processes,NaN,NaN,3.0,NaN,4.0,%,"Quarter ended September 30, 2023",Page 1 — Business Highlights,8
9,Narrative performance highlight,Business Highlights,Microsoft 365 Consumer subscribers,Productivity and Business Processes,76.70,NaN,NaN,NaN,NaN,million subscribers,"Quarter ended September 30, 2023",Page 1 — Business Highlights,9


In [ ]:
# ============================================================
# 8. Validate input counts and category diagnostics
# ============================================================

reference_schema_valid = reference_df.columns.tolist() == FIELDS

reference_record_count_valid = (
    len(reference_df) == EXPECTED_REFERENCE_RECORD_COUNT
)

extraction_record_count_valid = (
    len(extracted_df) == EXPECTED_EXTRACTION_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"].value_counts().to_dict()
)

extraction_category_counts = (
    extracted_df["Category"].value_counts(dropna=False).to_dict()
)

reference_category_counts_valid = (
    reference_category_counts == EXPECTED_CATEGORY_COUNTS
)

extraction_category_counts_valid = (
    extraction_category_counts == EXPECTED_CATEGORY_COUNTS
)

print("Reference schema valid:", reference_schema_valid)
print("Reference count valid:", reference_record_count_valid)
print("Extraction count valid:", extraction_record_count_valid)
print("Reference category counts valid:", reference_category_counts_valid)
print("Extraction category counts valid:", extraction_category_counts_valid)

if not reference_schema_valid:
    raise ValueError("The D6 reference schema is invalid.")

if not reference_record_count_valid:
    raise ValueError(
        f"Expected {EXPECTED_REFERENCE_RECORD_COUNT} reference records, "
        f"found {len(reference_df)}."
    )

if not reference_category_counts_valid:
    raise ValueError(
        "The fixed Stage 1 D6 reference category counts do not match "
        "the expected reference definition."
    )

Reference schema valid: True
Reference count valid: True
Extraction count valid: True
Reference category counts valid: True
Extraction category counts valid: True


In [ ]:
# ============================================================
# 9. Null handling and extracted field-type diagnostics
# ============================================================

def is_null(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


type_issue_rows = []
missing_mandatory_rows = []

for row_index, row in extracted_df.iterrows():

    for field in STRING_FIELDS:
        value = row[field]

        if is_null(value):
            missing_mandatory_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field
                }
            )

        elif not isinstance(value, str):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__
                }
            )

    for field in NUMERIC_FIELDS:
        value = row[field]

        if (
            not is_null(value)
            and (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
            )
        ):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__
                }
            )


type_issues_df = pd.DataFrame(type_issue_rows)
missing_mandatory_df = pd.DataFrame(missing_mandatory_rows)

field_types_valid = type_issues_df.empty
mandatory_fields_complete = missing_mandatory_df.empty

print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)
print("Type issues:", len(type_issues_df))
print("Missing mandatory values:", len(missing_mandatory_df))

if not type_issues_df.empty:
    display(type_issues_df)

if not missing_mandatory_df.empty:
    display(missing_mandatory_df)

Field types valid: True
Mandatory fields complete: False
Type issues: 0
Missing mandatory values: 24


,Record Index,Field
0,0,Business Area
1,1,Business Area
2,2,Business Area
3,3,Business Area
4,23,Business Area
5,24,Business Area
6,25,Business Area
7,26,Business Area
8,27,Business Area
9,31,Business Area


In [ ]:
# ============================================================
# 10. Controlled comparison normalisation
# ============================================================
# FROZEN FROM FINAL D6 BRANCH A VALIDATION — REUSE UNCHANGED.

def normalise_text(value):
    if is_null(value):
        return None

    text = unicodedata.normalize("NFKC", str(value))

    replacements = {
        "\u00a0": " ",
        "\u2007": " ",
        "\u202f": " ",
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u2018": "'",
        "\u2019": "'"
    }

    for source, target in replacements.items():
        text = text.replace(source, target)

    text = re.sub(r"\s+", " ", text)

    return text.strip().casefold()


STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "for", "from", "in",
    "is", "of", "on", "or", "the", "to", "was", "were", "with"
}


def comparison_tokens(value):
    text = normalise_text(value)

    if text is None:
        return set()

    text = re.sub(r"[^a-z0-9]+", " ", text)

    return {
        token
        for token in text.split()
        if token and token not in STOPWORDS
    }


def sequence_similarity(first, second):
    first_text = normalise_text(first)
    second_text = normalise_text(second)

    if first_text is None and second_text is None:
        return 1.0

    if first_text is None or second_text is None:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text
    ).ratio()


def jaccard_similarity(first, second):
    first_tokens = comparison_tokens(first)
    second_tokens = comparison_tokens(second)

    if not first_tokens and not second_tokens:
        return 1.0

    if not first_tokens or not second_tokens:
        return 0.0

    return (
        len(first_tokens & second_tokens)
        / len(first_tokens | second_tokens)
    )


def text_similarity(first, second):
    # Used for record alignment diagnostics only.
    # This is lexical/string similarity, not semantic similarity.
    return max(
        sequence_similarity(first, second),
        jaccard_similarity(first, second)
    )

In [ ]:
# ============================================================
# 11. Controlled D6 equivalence rules
# ============================================================
# FROZEN FROM FINAL D6 BRANCH A VALIDATION — REUSE UNCHANGED.

# IMPORTANT:
# These rules are document-level and source-grounded.
# Once frozen, the exact same rules must be used unchanged
# for D6 Branches A, B and C.


UNIT_EQUIVALENCE_MAP = {
    "usd billion": "usd billion",
    "usd billions": "usd billion",
    "billion usd": "usd billion",

    "usd millions": "usd millions",
    "usd millions; percent": "usd millions",

    "usd per share": "usd per share",
    "usd per share; percent": "usd per share",

    "percent": "percent",
    "percentage": "percent",

    "million subscribers": "million subscribers",

    "million shares": "million shares",
    "millions of shares": "million shares"
}


BUSINESS_AREA_EQUIVALENCE_MAP = {
    "total": "total",
    "corporate": "corporate",
    "microsoft": "corporate"
}


STATEMENT_EQUIVALENCE_MAP = {
    "quarterly results":
        "quarterly results and business highlights",

    "business highlights":
        "quarterly results and business highlights",

    "quarterly results and business highlights":
        "quarterly results and business highlights",

    "shareholder returns":
        "quarterly results and business highlights"
}


PERIOD_EQUIVALENCE_MAP = {
    "first quarter fiscal year 2024":
        "quarter ended september 30, 2023",

    "first quarter of fiscal year 2024":
        "quarter ended september 30, 2023",

    "quarter ended september 30, 2023":
        "quarter ended september 30, 2023",

    "three months ended september 30":
        "three months ended september 30, 2023",

    "three months ended september 30, 2023":
        "three months ended september 30, 2023",

    "september 30, 2023 and june 30, 2023":
        "september 30, 2023 and june 30, 2023"
}


def canonical_from_map(value, mapping):
    text = normalise_text(value)

    if text is None:
        return None

    return mapping.get(text, text)


def canonical_unit(value):
    return canonical_from_map(
        value,
        UNIT_EQUIVALENCE_MAP
    )


def canonical_business_area(value):
    return canonical_from_map(
        value,
        BUSINESS_AREA_EQUIVALENCE_MAP
    )


def canonical_statement(value):
    return canonical_from_map(
        value,
        STATEMENT_EQUIVALENCE_MAP
    )


def canonical_period(value):
    return canonical_from_map(
        value,
        PERIOD_EQUIVALENCE_MAP
    )


def canonical_metric(value):
    text = normalise_text(value)

    if text is None:
        return None

    text = text.replace(
        "weighted average shares outstanding - basic",
        "basic weighted average shares outstanding"
    )

    text = text.replace(
        "weighted average shares outstanding - diluted",
        "diluted weighted average shares outstanding"
    )

    return text

In [ ]:
# ============================================================
# 12A. Controlled Metric + Business Area equivalence
# ============================================================
# FROZEN FROM FINAL D6 BRANCH A VALIDATION — REUSE UNCHANGED.

# Some D6 observations distribute their semantic identity
# differently across Metric and Business Area.
#
# These explicitly verified pairs refer to the same source
# observation despite different field partitioning.
#
# IMPORTANT:
# These pairs are frozen at D6 document level and must be
# applied unchanged to Branches A, B and C.


METRIC_BUSINESS_EQUIVALENCE_RAW = [

    # Microsoft Cloud
    (
        ("Microsoft Cloud revenue", "Microsoft Cloud"),
        ("Revenue", "Microsoft Cloud")
    ),

    # Office Commercial
    (
        (
            "Office Commercial products and cloud services revenue",
            "Office Commercial"
        ),
        (
            "Revenue",
            "Office Commercial products and cloud services"
        )
    ),

    # Office 365 Commercial
    (
        (
            "Office 365 Commercial revenue",
            "Office 365 Commercial"
        ),
        (
            "Revenue growth",
            "Office 365 Commercial"
        )
    ),

    # Office Consumer
    (
        (
            "Office Consumer products and cloud services revenue",
            "Office Consumer"
        ),
        (
            "Revenue",
            "Office Consumer products and cloud services"
        )
    ),

    # LinkedIn
    (
        ("LinkedIn revenue", "LinkedIn"),
        ("Revenue", "LinkedIn")
    ),

    # Dynamics products and cloud services
    (
        (
            "Dynamics products and cloud services revenue",
            "Dynamics"
        ),
        (
            "Revenue",
            "Dynamics products and cloud services"
        )
    ),

    # Dynamics 365
    (
        ("Dynamics 365 revenue", "Dynamics 365"),
        ("Revenue growth", "Dynamics 365")
    ),

    # Server products and cloud services
    (
        (
            "Server products and cloud services revenue",
            "Server products and cloud services"
        ),
        (
            "Revenue",
            "Server products and cloud services"
        )
    ),

    # Azure
    (
        (
            "Azure and other cloud services revenue",
            "Azure and other cloud services"
        ),
        (
            "Revenue growth",
            "Azure and other cloud services"
        )
    ),

    # Windows
    (
        ("Windows revenue", "Windows"),
        ("Revenue", "Windows")
    ),

    # Windows OEM
    (
        ("Windows OEM revenue", "Windows OEM"),
        ("Revenue growth", "Windows OEM")
    ),

    # Windows Commercial
    (
        (
            "Windows Commercial products and cloud services revenue",
            "Windows Commercial"
        ),
        (
            "Revenue growth",
            "Windows Commercial products and cloud services"
        )
    ),

    # Devices
    (
        ("Devices revenue", "Devices"),
        ("Revenue", "Devices")
    ),

    # Xbox
    (
        (
            "Xbox content and services revenue",
            "Xbox content and services"
        ),
        (
            "Revenue",
            "Xbox content and services"
        )
    ),

    # Search and news advertising — narrative
    (
        (
            "Search and news advertising revenue excluding traffic acquisition costs",
            "Search and news advertising"
        ),
        (
            "Revenue excluding traffic acquisition costs",
            "Search and news advertising"
        )
    ),

    # Search and news advertising — reconciliation table
    (
        (
            "Revenue",
            "Search and news advertising excluding traffic acquisition costs"
        ),
        (
            "Revenue excluding traffic acquisition costs",
            "Search and news advertising"
        )
    )
]


def normalise_metric_business_pair(metric, business_area):
    return (
        canonical_metric(metric),
        canonical_business_area(business_area)
    )


METRIC_BUSINESS_EQUIVALENCE = {
    (
        normalise_metric_business_pair(
            reference_metric,
            reference_business
        ),
        normalise_metric_business_pair(
            extracted_metric,
            extracted_business
        )
    )
    for (
        (reference_metric, reference_business),
        (extracted_metric, extracted_business)
    )
    in METRIC_BUSINESS_EQUIVALENCE_RAW
}


def is_metric_business_pair_equivalent(
    reference_metric,
    reference_business,
    extracted_metric,
    extracted_business
):
    reference_pair = normalise_metric_business_pair(
        reference_metric,
        reference_business
    )

    extracted_pair = normalise_metric_business_pair(
        extracted_metric,
        extracted_business
    )

    return (
        reference_pair,
        extracted_pair
    ) in METRIC_BUSINESS_EQUIVALENCE

In [ ]:
# ============================================================
# 12. Numeric and exact-value comparison
# ============================================================
# FROZEN FROM FINAL D6 BRANCH A VALIDATION — REUSE UNCHANGED.

def numeric_value(value):
    if is_null(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        text = (
            value.strip()
            .replace(",", "")
            .replace("−", "-")
            .replace("–", "-")
        )

        if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
            return float(text)

    return None


def values_match(reference_value, extracted_value, tolerance=1e-9):

    # Both absent = agreement
    if is_null(reference_value) and is_null(extracted_value):
        return True

    # Only one absent = discrepancy
    if is_null(reference_value) or is_null(extracted_value):
        return False

    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if reference_number is not None and extracted_number is not None:
        return math.isclose(
            reference_number,
            extracted_number,
            rel_tol=tolerance,
            abs_tol=tolerance
        )

    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )

In [ ]:
# ============================================================
# 13. Prepare comparison copies and blocking keys
# ============================================================
# FROZEN FROM FINAL D6 BRANCH A VALIDATION — REUSE UNCHANGED.

reference_comparison_df = reference_df.copy(deep=True)
extracted_comparison_df = extracted_df.copy(deep=True)

reference_comparison_df["_reference_index"] = range(
    len(reference_comparison_df)
)

# _extraction_index already exists in extracted_comparison_df.

for dataframe in [
    reference_comparison_df,
    extracted_comparison_df
]:
    dataframe["_block_category"] = dataframe["Category"].map(normalise_text)
    dataframe["_block_source"] = dataframe["Source Location"].map(normalise_text)

    dataframe["_matching_block"] = list(
        zip(
            dataframe["_block_category"],
            dataframe["_block_source"]
        )
    )

print("Comparison copies prepared.")
print("Blocking fields:", BLOCK_FIELDS)

Comparison copies prepared.
Blocking fields: ['Category', 'Source Location']


In [ ]:
# ============================================================
# 14. Deterministic identity-based record-matching score
# ============================================================
# FROZEN FROM FINAL D6 BRANCH A VALIDATION — REUSE UNCHANGED.

def matching_score(reference_row, extracted_row):

    metric_score = text_similarity(
        canonical_metric(reference_row["Metric"]),
        canonical_metric(extracted_row["Metric"])
    )

    business_score = (
        1.0
        if canonical_business_area(reference_row["Business Area"])
        == canonical_business_area(extracted_row["Business Area"])
        else text_similarity(
            reference_row["Business Area"],
            extracted_row["Business Area"]
        )
    )

    statement_score = (
        1.0
        if canonical_statement(reference_row["Statement or Section"])
        == canonical_statement(extracted_row["Statement or Section"])
        else text_similarity(
            reference_row["Statement or Section"],
            extracted_row["Statement or Section"]
        )
    )

    period_score = (
        1.0
        if canonical_period(reference_row["Reporting Period"])
        == canonical_period(extracted_row["Reporting Period"])
        else text_similarity(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"]
        )
    )

    # IMPORTANT:
    # No extracted numerical value, Unit, GAAP change or
    # constant-currency outcome is used to establish record identity.
    total_score = (
        0.55 * metric_score
        + 0.25 * business_score
        + 0.15 * statement_score
        + 0.05 * period_score
    )

    return {
        "total": total_score,
        "metric": metric_score,
        "business_area": business_score,
        "statement": statement_score,
        "reporting_period": period_score
    }

In [ ]:
# ============================================================
# 15. One-to-one record alignment
# ============================================================
# FROZEN FROM FINAL D6 BRANCH A VALIDATION — REUSE UNCHANGED.

all_blocks = sorted(
    set(reference_comparison_df["_matching_block"])
    | set(extracted_comparison_df["_matching_block"]),
    key=str
)

matched_pairs = []

unmatched_reference_indices = set(
    reference_comparison_df["_reference_index"].tolist()
)

unmatched_extraction_indices = set(
    extracted_comparison_df["_extraction_index"].tolist()
)


for block in all_blocks:

    reference_block = reference_comparison_df.loc[
        reference_comparison_df["_matching_block"] == block
    ]

    extraction_block = extracted_comparison_df.loc[
        extracted_comparison_df["_matching_block"] == block
    ]

    if reference_block.empty or extraction_block.empty:
        continue

    reference_rows = [
        row
        for _, row in reference_block.iterrows()
    ]

    extraction_rows = [
        row
        for _, row in extraction_block.iterrows()
    ]

    score_matrix = []
    details_matrix = []

    for reference_row in reference_rows:

        score_row = []
        details_row = []

        for extracted_row in extraction_rows:
            details = matching_score(
                reference_row,
                extracted_row
            )

            score_row.append(details["total"])
            details_row.append(details)

        score_matrix.append(score_row)
        details_matrix.append(details_row)

    cost_matrix = [
        [1.0 - score for score in row]
        for row in score_matrix
    ]

    row_positions, column_positions = linear_sum_assignment(
        cost_matrix
    )

    for row_position, column_position in zip(
        row_positions,
        column_positions
    ):

        details = details_matrix[
            row_position
        ][
            column_position
        ]

        if details["total"] < MATCH_SCORE_THRESHOLD:
            continue

        reference_index = int(
            reference_rows[
                row_position
            ][
                "_reference_index"
            ]
        )

        extraction_index = int(
            extraction_rows[
                column_position
            ][
                "_extraction_index"
            ]
        )

        matched_pairs.append(
            {
                "reference_index": reference_index,
                "extraction_index": extraction_index,
                "matching_score": details["total"],
                "metric_matching_score": details["metric"],
                "business_area_matching_score":
                    details["business_area"],
                "statement_matching_score": details["statement"],
                "reporting_period_matching_score":
                    details["reporting_period"]
            }
        )

        unmatched_reference_indices.discard(
            reference_index
        )

        unmatched_extraction_indices.discard(
            extraction_index
        )


aligned_record_count = len(matched_pairs)
missing_record_count = len(unmatched_reference_indices)
unsupported_record_count = len(unmatched_extraction_indices)

print("Aligned records:", aligned_record_count)
print("Missing records:", missing_record_count)
print(
    "Unsupported/unmatched extracted records:",
    unsupported_record_count
)

Aligned records: 135
Missing records: 12
Unsupported/unmatched extracted records: 12


In [ ]:
# ============================================================
# 16. Missing and unsupported/unmatched record tables
# ============================================================

missing_records_df = reference_comparison_df.loc[
    reference_comparison_df["_reference_index"].isin(
        unmatched_reference_indices
    ),
    ["_reference_index"] + FIELDS
].copy()

unsupported_records_df = extracted_comparison_df.loc[
    extracted_comparison_df["_extraction_index"].isin(
        unmatched_extraction_indices
    ),
    ["_extraction_index"] + FIELDS
].copy()

print("Missing:", len(missing_records_df))
print("Unsupported/unmatched:", len(unsupported_records_df))

if not missing_records_df.empty:
    display(missing_records_df)

if not unsupported_records_df.empty:
    display(unsupported_records_df)

Missing: 12
Unsupported/unmatched: 12


,_reference_index,Category,Statement or Section,Metric,Business Area,Value 2023,Value 2022,GAAP YoY Change,Constant Currency Impact,Constant Currency YoY Change,Unit,Reporting Period,Source Location
33,33,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Office 365 Commercial,NaN,NaN,18.0,-1.0,17.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
34,34,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Office Consumer products and cloud services,NaN,NaN,3.0,1.0,4.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
36,36,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Dynamics products and cloud services,NaN,NaN,22.0,-1.0,21.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
37,37,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Dynamics 365,NaN,NaN,28.0,-2.0,26.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
38,38,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Server products and cloud services,NaN,NaN,21.0,0.0,21.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
39,39,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Azure and other cloud services,NaN,NaN,29.0,-1.0,28.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
40,40,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Windows,NaN,NaN,5.0,0.0,5.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
41,41,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Windows OEM,NaN,NaN,4.0,0.0,4.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
42,42,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Windows Commercial products and cloud services,NaN,NaN,8.0,0.0,8.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
43,43,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Revenue,Devices,NaN,NaN,-22.0,0.0,-22.0,percent,"Three months ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...


,_extraction_index,Category,Statement or Section,Metric,Business Area,Value 2023,Value 2022,GAAP YoY Change,Constant Currency Impact,Constant Currency YoY Change,Unit,Reporting Period,Source Location
31,31,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Microsoft Cloud,None,NaN,NaN,24.0,-1.0,23.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
32,32,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Office Commercial products and cloud services,None,NaN,NaN,15.0,-1.0,14.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
33,33,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Office 365 Commercial,None,NaN,NaN,18.0,-1.0,17.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
34,34,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Office Consumer products and cloud services,None,NaN,NaN,3.0,1.0,4.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
36,36,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Dynamics products and cloud services,None,NaN,NaN,22.0,-1.0,21.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
37,37,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Dynamics 365,None,NaN,NaN,28.0,-2.0,26.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
39,39,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Azure and other cloud services,None,NaN,NaN,29.0,-1.0,28.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
40,40,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Windows,None,NaN,NaN,5.0,0.0,5.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
41,41,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Windows OEM,None,NaN,NaN,4.0,0.0,4.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...
42,42,Selected product and service reconciliation,Selected Product and Service Revenue Constant ...,Windows Commercial products and cloud services,None,NaN,NaN,8.0,0.0,8.0,%,"Three Months Ended September 30, 2023",Page 3 — Selected Product and Service Revenue ...


In [ ]:
# ============================================================
# 17. Field-level comparison of aligned records
# ============================================================
# FROZEN FROM FINAL D6 BRANCH A VALIDATION — REUSE UNCHANGED.

def exact_text_match(first, second):
    return normalise_text(first) == normalise_text(second)


def mapped_text_match(first, second, canonical_function):
    return canonical_function(first) == canonical_function(second)


comparison_rows = []

for pair in matched_pairs:

    reference_row = reference_comparison_df.loc[
        reference_comparison_df["_reference_index"]
        == pair["reference_index"]
    ].iloc[0]

    extracted_row = extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"]
        == pair["extraction_index"]
    ].iloc[0]

    category_match = exact_text_match(
        reference_row["Category"],
        extracted_row["Category"]
    )

    statement_match = mapped_text_match(
        reference_row["Statement or Section"],
        extracted_row["Statement or Section"],
        canonical_statement
    )

    metric_similarity = text_similarity(
        canonical_metric(reference_row["Metric"]),
        canonical_metric(extracted_row["Metric"])
    )


    # First evaluate whether Metric + Business Area jointly
    # represent the same source observation.
    metric_business_pair_rule_applied = (
        is_metric_business_pair_equivalent(
            reference_row["Metric"],
            reference_row["Business Area"],
            extracted_row["Metric"],
            extracted_row["Business Area"]
        )
    )


    # Metric is correct if:
    # 1. it matches exactly after controlled normalisation, OR
    # 2. the predefined Metric + Business Area pair is equivalent.
    metric_match = (
        canonical_metric(reference_row["Metric"])
        == canonical_metric(extracted_row["Metric"])
        or metric_business_pair_rule_applied
    )


    # Business Area is correct if:
    # 1. it matches through the controlled Business Area map, OR
    # 2. the predefined Metric + Business Area pair is equivalent.
    business_area_match = (
        canonical_business_area(
            reference_row["Business Area"]
        )
        == canonical_business_area(
            extracted_row["Business Area"]
        )
        or metric_business_pair_rule_applied
    )

    value_2023_match = values_match(
        reference_row["Value 2023"],
        extracted_row["Value 2023"]
    )

    value_2022_match = values_match(
        reference_row["Value 2022"],
        extracted_row["Value 2022"]
    )

    gaap_change_match = values_match(
        reference_row["GAAP YoY Change"],
        extracted_row["GAAP YoY Change"]
    )

    constant_currency_impact_match = values_match(
        reference_row["Constant Currency Impact"],
        extracted_row["Constant Currency Impact"]
    )

    constant_currency_change_match = values_match(
        reference_row["Constant Currency YoY Change"],
        extracted_row["Constant Currency YoY Change"]
    )

    unit_match = mapped_text_match(
        reference_row["Unit"],
        extracted_row["Unit"],
        canonical_unit
    )

    reporting_period_match = mapped_text_match(
        reference_row["Reporting Period"],
        extracted_row["Reporting Period"],
        canonical_period
    )

    source_location_match = exact_text_match(
        reference_row["Source Location"],
        extracted_row["Source Location"]
    )

    field_matches = {
        "Category": category_match,
        "Statement or Section": statement_match,
        "Metric": metric_match,
        "Business Area": business_area_match,
        "Value 2023": value_2023_match,
        "Value 2022": value_2022_match,
        "GAAP YoY Change": gaap_change_match,
        "Constant Currency Impact":
            constant_currency_impact_match,
        "Constant Currency YoY Change":
            constant_currency_change_match,
        "Unit": unit_match,
        "Reporting Period": reporting_period_match,
        "Source Location": source_location_match
    }

    all_primary_fields_match = all(
        field_matches[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    all_mismatched_fields = [
        field
        for field, match in field_matches.items()
        if not match
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    output_row = {
        "Reference Index": pair["reference_index"],
        "Extraction Index": pair["extraction_index"],
        "Matching Score": pair["matching_score"],

        "Metric Matching Score":
            pair["metric_matching_score"],

        "Business Area Matching Score":
            pair["business_area_matching_score"],

        "Statement Matching Score":
            pair["statement_matching_score"],

        "Reporting Period Matching Score":
            pair["reporting_period_matching_score"],

        "Category":
            reference_row["Category"],

        "Reference Metric":
            reference_row["Metric"],

        "Extracted Metric":
            extracted_row["Metric"],

        "Metric Lexical Similarity":
            metric_similarity,

        "Metric-Business Pair Rule Applied":
            bool(metric_business_pair_rule_applied),

        "Fully Correct":
            bool(all_primary_fields_match),

        "all_mismatched_fields":
            ", ".join(all_mismatched_fields),

        "primary_mismatched_fields":
            ", ".join(primary_mismatched_fields)
    }



    for field in FIELDS:
        output_row[f"Reference {field}"] = reference_row[field]
        output_row[f"Extracted {field}"] = extracted_row[field]
        output_row[f"{field} Match"] = bool(field_matches[field])

    comparison_rows.append(output_row)


comparison_df = pd.DataFrame(comparison_rows)

print("Compared aligned records:", len(comparison_df))
print(
    "Fully correct:",
    int(comparison_df["Fully Correct"].sum())
)

display(comparison_df.head(10))

Compared aligned records: 135
Fully correct: 74


,Reference Index,Extraction Index,Matching Score,Metric Matching Score,Business Area Matching Score,Statement Matching Score,Reporting Period Matching Score,Category,Reference Metric,Extracted Metric,...,Constant Currency YoY Change Match,Reference Unit,Extracted Unit,Unit Match,Reference Reporting Period,Extracted Reporting Period,Reporting Period Match,Reference Source Location,Extracted Source Location,Source Location Match
0,71,71,1.000000,1.000000,1.0,1.0,1.0,Balance sheet,Cash and cash equivalents,Cash and cash equivalents,...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True
1,72,72,1.000000,1.000000,1.0,1.0,1.0,Balance sheet,Short-term investments,Short-term investments,...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True
2,73,73,1.000000,1.000000,1.0,1.0,1.0,Balance sheet,"Total cash, cash equivalents, and short-term i...","Total cash, cash equivalents, and short-term i...",...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True
3,74,74,0.780000,0.600000,1.0,1.0,1.0,Balance sheet,"Accounts receivable, net","Accounts receivable, net of allowance for doub...",...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True
4,75,75,1.000000,1.000000,1.0,1.0,1.0,Balance sheet,Inventories,Inventories,...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True
5,76,76,1.000000,1.000000,1.0,1.0,1.0,Balance sheet,Other current assets,Other current assets,...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True
6,77,77,1.000000,1.000000,1.0,1.0,1.0,Balance sheet,Total current assets,Total current assets,...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True
7,78,78,0.812195,0.658537,1.0,1.0,1.0,Balance sheet,"Property and equipment, net","Property and equipment, net of accumulated dep...",...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True
8,79,79,1.000000,1.000000,1.0,1.0,1.0,Balance sheet,Operating lease right-of-use assets,Operating lease right-of-use assets,...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True
9,80,80,1.000000,1.000000,1.0,1.0,1.0,Balance sheet,Equity investments,Equity investments,...,True,USD millions,USD millions,True,"September 30, 2023 and June 30, 2023","September 30, 2023 and June 30, 2023",True,Page 8 — Balance Sheets,Page 8 — Balance Sheets,True


In [ ]:
# ============================================================
# 18. Fully-correct and discrepant aligned records
# ============================================================

fully_correct_records_df = comparison_df.loc[
    comparison_df["Fully Correct"] == True
].copy()

discrepant_records_df = comparison_df.loc[
    comparison_df["Fully Correct"] == False
].copy()

print("Fully correct aligned records:", len(fully_correct_records_df))
print("Discrepant aligned records:", len(discrepant_records_df))

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Category",
                "Reference Metric",
                "Extracted Metric",
                "primary_mismatched_fields"
            ]
        ].head(30)
    )

Fully correct aligned records: 74
Discrepant aligned records: 61


,Reference Index,Extraction Index,Category,Reference Metric,Extracted Metric,primary_mismatched_fields
3,74,74,Balance sheet,"Accounts receivable, net","Accounts receivable, net of allowance for doub...",Metric
7,78,78,Balance sheet,"Property and equipment, net","Property and equipment, net of accumulated dep...",Metric
68,65,65,Comprehensive income statement,Net income,Net income,Unit
69,66,66,Comprehensive income statement,Net change related to derivatives,Net change related to derivatives,Unit
70,67,67,Comprehensive income statement,Net change related to investments,Net change related to investments,Unit
71,68,68,Comprehensive income statement,Translation adjustments and other,Translation adjustments and other,Unit
72,69,69,Comprehensive income statement,Other comprehensive loss,Other comprehensive loss,Unit
73,70,70,Comprehensive income statement,Comprehensive income,Comprehensive income,Unit
74,24,24,Financial performance reconciliation,Revenue,Revenue,"Business Area, Unit"
75,25,25,Financial performance reconciliation,Operating income,Operating Income,"Business Area, Unit"


In [ ]:
# ============================================================
# 19. Field discrepancy table
# ============================================================

field_discrepancy_rows = []

for _, row in comparison_df.iterrows():

    for field in FIELDS:

        if not bool(row[f"{field} Match"]):
            field_discrepancy_rows.append(
                {
                    "Reference Index":
                        int(row["Reference Index"]),
                    "Extraction Index":
                        int(row["Extraction Index"]),
                    "Category":
                        row["Category"],
                    "Reference Metric":
                        row["Reference Metric"],
                    "Extracted Metric":
                        row["Extracted Metric"],
                    "Field":
                        field,
                    "Reference Value":
                        row[f"Reference {field}"],
                    "Extracted Value":
                        row[f"Extracted {field}"]
                }
            )


field_discrepancies_df = pd.DataFrame(
    field_discrepancy_rows
)

print(
    "Field discrepancy count:",
    len(field_discrepancies_df)
)

if not field_discrepancies_df.empty:
    display(field_discrepancies_df)

Field discrepancy count: 108


,Reference Index,Extraction Index,Category,Reference Metric,Extracted Metric,Field,Reference Value,Extracted Value
0,74,74,Balance sheet,"Accounts receivable, net","Accounts receivable, net of allowance for doub...",Metric,"Accounts receivable, net","Accounts receivable, net of allowance for doub..."
1,78,78,Balance sheet,"Property and equipment, net","Property and equipment, net of accumulated dep...",Metric,"Property and equipment, net","Property and equipment, net of accumulated dep..."
2,65,65,Comprehensive income statement,Net income,Net income,Unit,USD millions,millions
3,66,66,Comprehensive income statement,Net change related to derivatives,Net change related to derivatives,Unit,USD millions,millions
4,67,67,Comprehensive income statement,Net change related to investments,Net change related to investments,Unit,USD millions,millions
...,...,...,...,...,...,...,...,...
103,35,43,Selected product and service reconciliation,Revenue,Devices,Metric,Revenue,Devices
104,35,43,Selected product and service reconciliation,Revenue,Devices,Business Area,LinkedIn,None
105,35,43,Selected product and service reconciliation,Revenue,Devices,GAAP YoY Change,8.0,-22.0
106,35,43,Selected product and service reconciliation,Revenue,Devices,Constant Currency YoY Change,8.0,-22.0


In [ ]:
# ============================================================
# 20. Field-level accuracy among aligned records
# ============================================================

field_accuracy_rows = []

for field in FIELDS:

    correct_count = int(
        comparison_df[f"{field} Match"].sum()
    )

    aligned_count = len(comparison_df)

    field_accuracy_rows.append(
        {
            "Field": field,
            "Correct Records": correct_count,
            "Aligned Records": aligned_count,
            "Accuracy": (
                correct_count / aligned_count
                if aligned_count > 0
                else None
            )
        }
    )


field_accuracy_df = pd.DataFrame(
    field_accuracy_rows
)

field_accuracy_dictionary = {
    row["Field"]:
        (
            float(row["Accuracy"])
            if pd.notna(row["Accuracy"])
            else None
        )
    for _, row in field_accuracy_df.iterrows()
}

display(field_accuracy_df)

,Field,Correct Records,Aligned Records,Accuracy
0,Category,135,135,1.000000
1,Statement or Section,135,135,1.000000
2,Metric,120,135,0.888889
3,Business Area,108,135,0.800000
4,Value 2023,135,135,1.000000
5,Value 2022,135,135,1.000000
6,GAAP YoY Change,132,135,0.977778
7,Constant Currency Impact,133,135,0.985185
8,Constant Currency YoY Change,132,135,0.977778
9,Unit,77,135,0.570370


In [ ]:
# ============================================================
# 21. Record-level metrics
# ============================================================

fully_correct_record_count = int(
    comparison_df["Fully Correct"].sum()
)

discrepant_record_count = (
    aligned_record_count
    - fully_correct_record_count
)

completeness = (
    aligned_record_count / len(reference_df)
    if len(reference_df) > 0
    else 0.0
)

missing_rate = (
    missing_record_count / len(reference_df)
    if len(reference_df) > 0
    else 0.0
)

record_precision_exact = (
    fully_correct_record_count / len(extracted_df)
    if len(extracted_df) > 0
    else 0.0
)

record_recall_exact = (
    fully_correct_record_count / len(reference_df)
    if len(reference_df) > 0
    else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if (
        record_precision_exact + record_recall_exact
        > 0
    )
    else 0.0
)

unsupported_rate = (
    unsupported_record_count / len(extracted_df)
    if len(extracted_df) > 0
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count / aligned_record_count
    if aligned_record_count > 0
    else 0.0
)

overall_primary_field_accuracy = (
    sum(
        int(
            comparison_df[
                f"{field} Match"
            ].sum()
        )
        for field in PRIMARY_CORRECTNESS_FIELDS
    )
    / (
        len(comparison_df)
        * len(PRIMARY_CORRECTNESS_FIELDS)
    )
    if len(comparison_df) > 0
    else None
)

schema_error_rate = (
    len(schema_issues_df) / len(extracted_df)
    if len(extracted_df) > 0
    else 0.0
)

print("Fully correct records:", fully_correct_record_count)
print("Discrepant records:", discrepant_record_count)
print("Completeness:", completeness)
print("Exact record precision:", record_precision_exact)
print("Exact record recall:", record_recall_exact)
print("Exact record F1:", record_f1_exact)
print("Missing rate:", missing_rate)
print("Unsupported rate:", unsupported_rate)
print(
    "Discrepancy rate among aligned:",
    discrepancy_rate_among_aligned
)
print(
    "Overall primary field accuracy:",
    overall_primary_field_accuracy
)
print("Schema error rate:", schema_error_rate)

Fully correct records: 74
Discrepant records: 61
Completeness: 0.9183673469387755
Exact record precision: 0.5034013605442177
Exact record recall: 0.5034013605442177
Exact record F1: 0.5034013605442177
Missing rate: 0.08163265306122448
Unsupported rate: 0.08163265306122448
Discrepancy rate among aligned: 0.45185185185185184
Overall primary field accuracy: 0.9333333333333333
Schema error rate: 0.0


In [ ]:
# ============================================================
# 22. Category-level metrics
# ============================================================

category_metric_rows = []

for category, expected_count in EXPECTED_CATEGORY_COUNTS.items():

    category_comparison = comparison_df.loc[
        comparison_df["Category"] == category
    ]

    extracted_category_count = int(
        (
            extracted_df["Category"]
            == category
        ).sum()
    )

    aligned_count = len(category_comparison)

    fully_correct_count = int(
        category_comparison[
            "Fully Correct"
        ].sum()
    )

    discrepant_count = (
        aligned_count - fully_correct_count
    )

    category_completeness = (
        aligned_count / expected_count
        if expected_count > 0
        else None
    )

    category_precision_exact = (
        fully_correct_count
        / extracted_category_count
        if extracted_category_count > 0
        else 0.0
    )

    category_recall_exact = (
        fully_correct_count
        / expected_count
        if expected_count > 0
        else 0.0
    )

    category_f1_exact = (
        2
        * category_precision_exact
        * category_recall_exact
        / (
            category_precision_exact
            + category_recall_exact
        )
        if (
            category_precision_exact
            + category_recall_exact
            > 0
        )
        else 0.0
    )

    category_metric_rows.append(
        {
            "Category": category,
            "Expected Records": expected_count,
            "Extracted Records": extracted_category_count,
            "Aligned Records": aligned_count,
            "Fully Correct Records": fully_correct_count,
            "Discrepant Records": discrepant_count,
            "Completeness": category_completeness,
            "Record Precision Exact":
                category_precision_exact,
            "Record Recall Exact":
                category_recall_exact,
            "Record F1 Exact":
                category_f1_exact
        }
    )


category_metrics_df = pd.DataFrame(
    category_metric_rows
)

display(category_metrics_df)

,Category,Expected Records,Extracted Records,Aligned Records,Fully Correct Records,Discrepant Records,Completeness,Record Precision Exact,Record Recall Exact,Record F1 Exact
0,Narrative performance highlight,24,24,24,0,24,1.0,0.000000,0.000000,0.000000
1,Financial performance reconciliation,4,4,4,0,4,1.0,0.000000,0.000000,0.000000
2,Segment revenue reconciliation,3,3,3,0,3,1.0,0.000000,0.000000,0.000000
3,Selected product and service reconciliation,15,15,3,0,3,0.2,0.000000,0.000000,0.000000
4,Income statement,19,19,19,0,19,1.0,0.000000,0.000000,0.000000
5,Comprehensive income statement,6,6,6,0,6,1.0,0.000000,0.000000,0.000000
6,Balance sheet,34,34,34,32,2,1.0,0.941176,0.941176,0.941176
7,Cash flow statement,34,34,34,34,0,1.0,1.000000,1.000000,1.000000
8,Segment revenue and operating income,8,8,8,8,0,1.0,1.000000,1.000000,1.000000


In [ ]:
# ============================================================
# 23. Define schema validity independently from completeness
# ============================================================
# Technical/schema validity is separate from expected record-count
# and category-count agreement.
#
# IMPORTANT:
# Branch C structure-check diagnostics are the authoritative source
# for technical schema validity and mandatory-field completeness.
# Validation-side checks are retained only as secondary diagnostics.

schema_diagnostics = {
    "valid_json":
        bool(valid_json),

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_correct":
        bool(document_id_correct),

    "branch_correct":
        bool(branch_correct),

    "records_is_list":
        bool(records_is_list),

    "record_schema_valid":
        bool(record_schema_valid),

    "field_types_valid":
        bool(field_types_valid),

    "records_with_structure_issues":
        int(
            len(schema_issues_df)
        ),

    "records_with_type_issues":
        int(
            len(type_issues_df)
        ),

    "branch_C_structure_valid":
        bool(
            branch_structure_valid
        ),

    "branch_C_record_schema_valid":
        branch_structure_check.get(
            "record_schema_valid"
        ),

    "branch_C_field_types_valid":
        branch_structure_check.get(
            "field_types_valid"
        )
}


schema_validity = all([
    schema_diagnostics[
        "valid_json"
    ],

    schema_diagnostics[
        "top_level_object_valid"
    ],

    schema_diagnostics[
        "document_id_correct"
    ],

    schema_diagnostics[
        "branch_correct"
    ],

    schema_diagnostics[
        "records_is_list"
    ],

    schema_diagnostics[
        "record_schema_valid"
    ],

    schema_diagnostics[
        "field_types_valid"
    ],

    schema_diagnostics[
        "branch_C_structure_valid"
    ]
])


schema_diagnostics[
    "schema_validity"
] = bool(
    schema_validity
)


# ------------------------------------------------------------
# Branch C content diagnostics
# ------------------------------------------------------------

branch_C_content_diagnostics = (
    branch_structure_check.get(
        "content_diagnostics",
        {}
    )
    or {}
)


branch_C_mandatory_fields_complete = (
    branch_C_content_diagnostics.get(
        "mandatory_fields_complete"
    )
)


# ------------------------------------------------------------
# Content diagnostics
# ------------------------------------------------------------
#
# The Branch C structure check is authoritative for technical
# mandatory-field completeness.
#
# The validation-side result is retained separately only as an
# auxiliary diagnostic and does not determine schema validity.
# ------------------------------------------------------------

content_diagnostics = {

    "reference_record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "reference_category_counts_valid":
        bool(
            reference_category_counts_valid
        ),

    "extraction_record_count_valid":
        bool(
            extraction_record_count_valid
        ),

    "extraction_category_counts_valid":
        bool(
            extraction_category_counts_valid
        ),

    # Authoritative execution-side result.
    "mandatory_fields_complete":
        (
            bool(
                branch_C_mandatory_fields_complete
            )
            if branch_C_mandatory_fields_complete
            is not None
            else bool(
                mandatory_fields_complete
            )
        ),

    # Secondary validation-side diagnostic retained
    # for transparency only.
    "validation_side_mandatory_fields_complete":
        bool(
            mandatory_fields_complete
        ),

    "branch_C_mandatory_fields_complete":
        branch_C_mandatory_fields_complete,

    "branch_C_scope_complete":
        branch_structure_check.get(
            "scope_complete"
        ),

    "branch_C_content_diagnostics":
        branch_C_content_diagnostics
}


print(
    json.dumps(
        schema_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)


print(
    "\nContent diagnostics kept separate from schema:"
)

print(
    json.dumps(
        content_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)

{
  "valid_json": true,
  "top_level_object_valid": true,
  "document_id_correct": true,
  "branch_correct": true,
  "records_is_list": true,
  "record_schema_valid": true,
  "field_types_valid": true,
  "records_with_structure_issues": 0,
  "records_with_type_issues": 0,
  "branch_C_structure_valid": true,
  "branch_C_record_schema_valid": true,
  "branch_C_field_types_valid": true,
  "schema_validity": true
}

Content diagnostics kept separate from schema:
{
  "reference_record_count_valid": true,
  "reference_category_counts_valid": true,
  "extraction_record_count_valid": true,
  "extraction_category_counts_valid": true,
  "mandatory_fields_complete": true,
  "validation_side_mandatory_fields_complete": false,
  "branch_C_mandatory_fields_complete": true,
  "branch_C_scope_complete": true,
  "branch_C_content_diagnostics": {
    "expected_record_count": 147,
    "observed_record_count": 147,
    "record_count_matches_reference": true,
    "expected_category_counts": {
      "Narrat

In [ ]:
# ============================================================
# 24. Validation summary table
# ============================================================

validation_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Reference records",
            "Value": len(reference_df)
        },
        {
            "Metric": "Extracted records",
            "Value": len(extracted_df)
        },
        {
            "Metric": "Aligned records",
            "Value": aligned_record_count
        },
        {
            "Metric": "Fully correct records",
            "Value": fully_correct_record_count
        },
        {
            "Metric": "Discrepant records",
            "Value": discrepant_record_count
        },
        {
            "Metric": "Missing records",
            "Value": missing_record_count
        },
        {
            "Metric": "Unsupported extracted records",
            "Value": unsupported_record_count
        },
        {
            "Metric": "Completeness",
            "Value": completeness
        },
        {
            "Metric": "Record precision exact",
            "Value": record_precision_exact
        },
        {
            "Metric": "Record recall exact",
            "Value": record_recall_exact
        },
        {
            "Metric": "Record F1 exact",
            "Value": record_f1_exact
        },
        {
            "Metric": "Overall primary field accuracy",
            "Value": overall_primary_field_accuracy
        },
        {
            "Metric": "Schema validity",
            "Value": schema_validity
        },
        {
            "Metric": "Schema issue count",
            "Value": len(schema_issues_df)
        }
    ]
)

display(validation_summary_df)

,Metric,Value
0,Reference records,147
1,Extracted records,147
2,Aligned records,135
3,Fully correct records,74
4,Discrepant records,61
5,Missing records,12
6,Unsupported extracted records,12
7,Completeness,0.918367
8,Record precision exact,0.503401
9,Record recall exact,0.503401


In [ ]:
# ============================================================
# 25. Preserve Branch C normalisation-integrity diagnostics
# ============================================================
# Stage 2 representation integrity is recorded independently of
# Stage 4 extraction correctness.

with NORMALISATION_CHECK_PATH.open("r", encoding="utf-8-sig") as file:
    normalisation_integrity = json.load(file)

if normalisation_integrity.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Normalisation-integrity document_id does not match D6."
    )

if normalisation_integrity.get("branch") != BRANCH:
    raise ValueError(
        "Normalisation-integrity branch does not match Branch C."
    )

if normalisation_integrity.get("parent_branch") != PARENT_BRANCH:
    raise ValueError(
        "Normalisation-integrity parent branch does not match Branch B."
    )

representation_integrity = {
    "parent_branch":
        normalisation_integrity.get("parent_branch"),

    "parent_equivalence_passed":
        bool(
            normalisation_integrity.get(
                "parent_equivalence_passed",
                False
            )
        ),

    "normalisation_integrity_passed":
        bool(
            normalisation_integrity.get(
                "normalisation_integrity_passed",
                False
            )
        ),

    "page_sequence_preserved":
        normalisation_integrity.get(
            "page_sequence_preserved"
        ),

    "deterministic_representation_verified":
        normalisation_integrity.get(
            "deterministic_representation_verified"
        ),

    "all_expected_components_preserved":
        normalisation_integrity.get(
            "all_expected_components_preserved"
        ),

    "numeric_values_preserved":
        normalisation_integrity.get(
            "numeric_values_preserved"
        ),

    "negative_parentheses_preserved":
        normalisation_integrity.get(
            "negative_parentheses_preserved"
        ),

    "zero_value_count_preserved":
        normalisation_integrity.get(
            "zero_value_count_preserved"
        ),

    "all_scope_pages_present":
        normalisation_integrity.get(
            "all_scope_pages_present"
        ),

    "complete_10_page_representation_retained":
        normalisation_integrity.get(
            "complete_10_page_representation_retained"
        ),

    "same_complete_representation_used_for_all_parts":
        normalisation_integrity.get(
            "same_complete_representation_used_for_all_parts"
        ),

    "part_specific_source_filtering_applied":
        normalisation_integrity.get(
            "part_specific_source_filtering_applied"
        ),

    "page_cropping_applied":
        normalisation_integrity.get(
            "page_cropping_applied"
        ),

    "page_removal_applied":
        normalisation_integrity.get(
            "page_removal_applied"
        ),

    "unicode_nfkc_normalisation_applied":
        normalisation_integrity.get(
            "unicode_nfkc_normalisation_applied"
        ),

    "unicode_space_standardisation_applied":
        normalisation_integrity.get(
            "unicode_space_standardisation_applied"
        ),

    "apostrophe_standardisation_applied":
        normalisation_integrity.get(
            "apostrophe_standardisation_applied"
        ),

    "dash_and_minus_standardisation_applied":
        normalisation_integrity.get(
            "dash_and_minus_standardisation_applied"
        ),

    "soft_hyphen_removal_applied":
        normalisation_integrity.get(
            "soft_hyphen_removal_applied"
        ),

    "semantic_harmonisation_applied":
        normalisation_integrity.get(
            "semantic_harmonisation_applied"
        ),

    "semantic_rewriting_applied":
        normalisation_integrity.get(
            "semantic_rewriting_applied"
        ),

    "unit_conversion_applied":
        normalisation_integrity.get(
            "unit_conversion_applied"
        ),

    "numeric_calculation_applied":
        normalisation_integrity.get(
            "numeric_calculation_applied"
        ),

    "manual_correction_applied":
        normalisation_integrity.get(
            "manual_correction_applied"
        ),

    "reference_values_used_for_transformation":
        normalisation_integrity.get(
            "reference_values_used_for_transformation"
        ),

    "numeric_token_preservation":
        normalisation_integrity.get(
            "numeric_token_preservation"
        ),

    "component_checks":
        normalisation_integrity.get(
            "component_checks"
        )
}

print("Branch C representation integrity:")
print(json.dumps(
    representation_integrity,
    indent=2,
    ensure_ascii=False
))

Branch C representation integrity:
{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "all_expected_components_preserved": true,
  "numeric_values_preserved": true,
  "negative_parentheses_preserved": true,
  "zero_value_count_preserved": true,
  "all_scope_pages_present": true,
  "complete_10_page_representation_retained": true,
  "same_complete_representation_used_for_all_parts": true,
  "part_specific_source_filtering_applied": false,
  "page_cropping_applied": false,
  "page_removal_applied": false,
  "unicode_nfkc_normalisation_applied": true,
  "unicode_space_standardisation_applied": true,
  "apostrophe_standardisation_applied": true,
  "dash_and_minus_standardisation_applied": true,
  "soft_hyphen_removal_applied": true,
  "semantic_harmonisation_applied": false,
  "semantic_rewriting_applied": false,
  "unit_conversion_applied": false,
  "

In [ ]:
# ============================================================
# 26. Create reproducible Branch C validation metrics
# ============================================================

category_metrics_dictionary = {
    row["Category"]: {
        "expected_records":
            int(row["Expected Records"]),
        "extracted_records":
            int(row["Extracted Records"]),
        "aligned_records":
            int(row["Aligned Records"]),
        "fully_correct_records":
            int(row["Fully Correct Records"]),
        "discrepant_records":
            int(row["Discrepant Records"]),
        "completeness":
            float(row["Completeness"]),
        "record_precision_exact":
            float(row["Record Precision Exact"]),
        "record_recall_exact":
            float(row["Record Recall Exact"]),
        "record_f1_exact":
            float(row["Record F1 Exact"])
    }
    for _, row in category_metrics_df.iterrows()
}

VALIDATION_METRICS = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,
    "input_representation": INPUT_REPRESENTATION,

    "reference_records": int(len(reference_df)),
    "extracted_records": int(len(extracted_df)),
    "aligned_records": int(aligned_record_count),
    "fully_correct_records": int(fully_correct_record_count),
    "discrepant_records": int(discrepant_record_count),
    "missing_records": int(missing_record_count),

    # Dissertation-facing term. These are unmatched extracted
    # observations relative to the fixed Stage 1 reference scope.
    "unsupported_records": int(unsupported_record_count),
    "unsupported_extracted_records": int(unsupported_record_count),

    "completeness": round(completeness, 4),
    "missing_rate": round(missing_rate, 4),
    "record_precision_exact": round(record_precision_exact, 4),
    "record_recall_exact": round(record_recall_exact, 4),
    "record_f1_exact": round(record_f1_exact, 4),
    "unsupported_rate": round(unsupported_rate, 4),
    "discrepancy_rate_among_aligned":
        round(discrepancy_rate_among_aligned, 4),

    "overall_primary_field_accuracy":
        (
            round(overall_primary_field_accuracy, 4)
            if overall_primary_field_accuracy is not None
            else None
        ),

    "field_accuracy_among_aligned": {
        field:
            (
                round(value, 4)
                if value is not None
                else None
            )
        for field, value
        in field_accuracy_dictionary.items()
    },

    "schema_validity": bool(schema_validity),
    "schema_diagnostics": schema_diagnostics,
    "content_diagnostics": content_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "matching_rules": {
        "blocking_fields": BLOCK_FIELDS,
        "one_to_one_assignment":
            "Hungarian linear-sum assignment",
        "matching_score_threshold":
            MATCH_SCORE_THRESHOLD,
        "matching_score_weights": {
            "metric": 0.55,
            "business_area": 0.25,
            "statement_or_section": 0.15,
            "reporting_period": 0.05
        },
        "value_2023_used_for_alignment": False,
        "value_2022_used_for_alignment": False,
        "unit_used_for_alignment": False,
        "gaap_change_used_for_alignment": False,
        "constant_currency_fields_used_for_alignment": False
    },

    "comparison_rules": {
        "raw_extraction_modified": False,
        "manual_correction_applied": False,
        "unexpected_fields_copied_to_expected_fields": False,
        "comparison_normalisation_scope":
            "Comparison copies only",
        "null_comparison":
            "None and pandas NaN treated as equivalent absence",
        "numeric_comparison":
            "Exact numeric equality after deterministic parsing",
        "metric":
            (
                "Normalised exact matching supplemented by "
                "predefined source-grounded Metric + Business Area "
                "pair equivalence where observation identity is "
                "distributed across both fields"
            ),
        "metric_business_pair_equivalence":
            (
                "Controlled document-level equivalence for verified "
                "alternative partitions of the same source observation"
            ),
        "metric_business_equivalence_rules_frozen_across_branches":
            True,
        "metric_lexical_similarity":
            "Diagnostic only; not used for field correctness",
        "business_area":
            (
                "Controlled source-grounded equivalence map, "
                "supplemented by predefined Metric + Business Area "
                "pair equivalence"
            ),
        "statement_or_section":
            (
                "Controlled source-grounded equivalence map; "
                "Quarterly results, Business Highlights and "
                "Shareholder returns are treated as equivalent "
                "within the predefined D6 source scope"
            ),
        "unit":
            (
                "Controlled source-grounded equivalence map; "
                "equivalent ordering variants are canonicalised, "
                "and reconciliation compound units are reduced to "
                "the base value unit because percentage changes are "
                "represented in dedicated schema fields"
            ),
        "reporting_period":
            (
                "Controlled source-grounded canonical equivalence "
                "for fiscal-quarter and quarter-end formulations"
            ),
        "source_location":
            "Normalised exact correctness after alignment",
        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,
        "d6_equivalence_rules_frozen_across_branches":
            True
    },

    "category_metrics":
        category_metrics_dictionary,

    "normalisation_note":
        (
            "Stage 4 comparison normalisation is applied only to "
            "comparison copies using rules frozen in D6 Branch A. "
            "It is distinct from Branch C deterministic input "
            "normalisation; the preserved Branch C extraction "
            "is not modified."
        ),

    "input_provenance": {
        "reference_file": REFERENCE_PATH.name,
        "reference_sha256": REFERENCE_SHA256,
        "combined_extraction_file": EXTRACTION_PATH.name,
        "combined_extraction_sha256": EXTRACTION_SHA256,
        "structure_check_file": STRUCTURE_CHECK_PATH.name,
        "structure_check_sha256": STRUCTURE_CHECK_SHA256,
        "normalisation_check_file":
            NORMALISATION_CHECK_PATH.name,
        "normalisation_check_sha256":
            NORMALISATION_CHECK_SHA256,
        "experiment_summary_file":
            (
                EXPERIMENT_SUMMARY_PATH.name
                if EXPERIMENT_SUMMARY_PATH is not None
                else None
            ),
        "experiment_summary_sha256":
            EXPERIMENT_SUMMARY_SHA256
    },

    "reference_dataset_branch_independent": True,
    "comparison_rules_frozen_from_branch_A": True,
    "validation_timestamp": datetime.now().isoformat()
}

print(json.dumps(
    VALIDATION_METRICS,
    ensure_ascii=False,
    indent=2
))

{
  "document_id": "D6",
  "document_name": "Microsoft FY24 Q1 Press Release",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "input_representation": "Complete deterministically normalised structural Markdown",
  "reference_records": 147,
  "extracted_records": 147,
  "aligned_records": 135,
  "fully_correct_records": 74,
  "discrepant_records": 61,
  "missing_records": 12,
  "unsupported_records": 12,
  "unsupported_extracted_records": 12,
  "completeness": 0.9184,
  "missing_rate": 0.0816,
  "record_precision_exact": 0.5034,
  "record_recall_exact": 0.5034,
  "record_f1_exact": 0.5034,
  "unsupported_rate": 0.0816,
  "discrepancy_rate_among_aligned": 0.4519,
  "overall_primary_field_accuracy": 0.9333,
  "field_accuracy_among_aligned": {
    "Category": 1.0,
    "Statement or Section": 1.0,
    "Metric": 0.8889,
    "Business Area": 0.8,
    "Value 2023": 1.0,
    "Value 2022": 1.0,
    "GAAP YoY Change": 0.9778,
    "Constant Currency Impac

In [ ]:
# ============================================================
# 27. Validation metadata and conclusion
# ============================================================

validation_status = (
    "Completed without discrepancies"
    if (
        fully_correct_record_count
        == EXPECTED_REFERENCE_RECORD_COUNT
        and missing_record_count == 0
        and unsupported_record_count == 0
        and schema_validity
    )
    else "Completed with discrepancies"
)

VALIDATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,
    "validation_type":
        "Deterministic comparison against the fixed D6 Stage 1 reference dataset",
    "raw_extraction_modified": False,
    "manual_correction_applied": False,
    "schema_errors_preserved": True,
    "comparison_normalisation_scope":
        "Comparison copies only",
    "matching_outcome_values_used": False,
    "automatic_unmatched_label":
        "unsupported/unmatched; not automatically hallucinated",
    "comparison_rules_frozen_from_branch_A": True,
    "created_at": datetime.now().isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "notes": (
        "D6 Branch C uses the same five-part execution protocol as "
        "Branches A and B. Each part is grounded in the same complete "
        "deterministically normalised 10-page Markdown representation. "
        "Validation is performed on the combined parsed extraction "
        "using the final Branch A alignment and comparison rules "
        "unchanged."
    )
}

VALIDATION_CONCLUSION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "validation_status": validation_status,
    "reference_records": int(len(reference_df)),
    "extracted_records": int(len(extracted_df)),
    "aligned_records": int(aligned_record_count),
    "fully_correct_records": int(fully_correct_record_count),
    "discrepant_records": int(discrepant_record_count),
    "missing_records": int(missing_record_count),
    "unsupported_records": int(unsupported_record_count),
    "completeness": round(completeness, 4),
    "record_precision_exact": round(record_precision_exact, 4),
    "record_recall_exact": round(record_recall_exact, 4),
    "record_f1_exact": round(record_f1_exact, 4),
    "overall_primary_field_accuracy":
        (
            round(overall_primary_field_accuracy, 4)
            if overall_primary_field_accuracy is not None
            else None
        ),
    "schema_valid": bool(schema_validity),
    "normalisation_integrity_passed":
        representation_integrity[
            "normalisation_integrity_passed"
        ],
    "notes": (
        "Schema validity, Branch C normalisation integrity, scope "
        "completeness, record correspondence, and field-level "
        "correctness are reported as separate outcomes."
    )
}

print(json.dumps(
    VALIDATION_CONCLUSION,
    ensure_ascii=False,
    indent=2
))

{
  "document_id": "D6",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "validation_status": "Completed with discrepancies",
  "reference_records": 147,
  "extracted_records": 147,
  "aligned_records": 135,
  "fully_correct_records": 74,
  "discrepant_records": 61,
  "missing_records": 12,
  "unsupported_records": 12,
  "completeness": 0.9184,
  "record_precision_exact": 0.5034,
  "record_recall_exact": 0.5034,
  "record_f1_exact": 0.5034,
  "overall_primary_field_accuracy": 0.9333,
  "schema_valid": true,
  "normalisation_integrity_passed": true,
  "notes": "Schema validity, Branch C normalisation integrity, scope completeness, record correspondence, and field-level correctness are reported as separate outcomes."
}


In [ ]:
# ============================================================
# 28. Define and export Branch C validation outputs
# ============================================================

DETAILED_PATH = OUTPUT_DIR / "D6_branch_C_validation_detailed.csv"
FULLY_CORRECT_RECORDS_PATH = (
    OUTPUT_DIR / "D6_branch_C_fully_correct_records.csv"
)
DISCREPANT_RECORDS_PATH = (
    OUTPUT_DIR / "D6_branch_C_discrepant_records.csv"
)
MISSING_RECORDS_PATH = (
    OUTPUT_DIR / "D6_branch_C_missing_records.csv"
)
UNSUPPORTED_RECORDS_PATH = (
    OUTPUT_DIR / "D6_branch_C_unsupported_records.csv"
)
SCHEMA_ISSUES_PATH = (
    OUTPUT_DIR / "D6_branch_C_schema_issues.csv"
)
TYPE_ISSUES_PATH = (
    OUTPUT_DIR / "D6_branch_C_type_issues.csv"
)
FIELD_DISCREPANCIES_PATH = (
    OUTPUT_DIR / "D6_branch_C_field_discrepancies.csv"
)
FIELD_ACCURACY_PATH = (
    OUTPUT_DIR / "D6_branch_C_field_error_summary.csv"
)
CATEGORY_METRICS_PATH = (
    OUTPUT_DIR / "D6_branch_C_category_metrics.csv"
)
VALIDATION_SUMMARY_CSV_PATH = (
    OUTPUT_DIR / "D6_branch_C_validation_summary.csv"
)
VALIDATION_SUMMARY_JSON_PATH = (
    OUTPUT_DIR / "D6_branch_C_validation_summary.json"
)
VALIDATION_METADATA_PATH = (
    OUTPUT_DIR / "D6_branch_C_validation_metadata.json"
)
VALIDATION_CONCLUSION_PATH = (
    OUTPUT_DIR / "D6_branch_C_validation_conclusion.json"
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

discrepant_records_df.to_csv(
    DISCREPANT_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

missing_records_df.to_csv(
    MISSING_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

unsupported_records_df.to_csv(
    UNSUPPORTED_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

schema_issues_df.to_csv(
    SCHEMA_ISSUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

type_issues_df.to_csv(
    TYPE_ISSUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_discrepancies_df.to_csv(
    FIELD_DISCREPANCIES_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_accuracy_df.to_csv(
    FIELD_ACCURACY_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

validation_summary_df.to_csv(
    VALIDATION_SUMMARY_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

VALIDATION_SUMMARY_JSON_PATH.write_text(
    json.dumps(
        VALIDATION_METRICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

VALIDATION_METADATA_PATH.write_text(
    json.dumps(
        VALIDATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

VALIDATION_CONCLUSION_PATH.write_text(
    json.dumps(
        VALIDATION_CONCLUSION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("Validation artefacts saved.")

Validation artefacts saved.


In [ ]:
# ============================================================
# 29. Final validation consistency checks
# ============================================================

if not reference_schema_valid:
    raise AssertionError(
        "Reference schema validation failed."
    )

if not reference_record_count_valid:
    raise AssertionError(
        "Reference record-count validation failed."
    )

if not reference_category_counts_valid:
    raise AssertionError(
        "Reference category-count validation failed."
    )

if not field_types_valid:
    raise AssertionError(
        "The extracted comparison fields contain invalid data types."
    )

if not representation_integrity["normalisation_integrity_passed"]:
    raise AssertionError(
        "Branch C representation-integrity checks did not pass."
    )

if aligned_record_count + missing_record_count != len(reference_df):
    raise AssertionError(
        "Reference-record accounting is inconsistent."
    )

if aligned_record_count + unsupported_record_count != len(extracted_df):
    raise AssertionError(
        "Extraction-record accounting is inconsistent."
    )

if fully_correct_record_count + discrepant_record_count != aligned_record_count:
    raise AssertionError(
        "Aligned-record correctness accounting is inconsistent."
    )

print("Validation status:", validation_status)
print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_df))
print("Aligned records:", aligned_record_count)
print("Missing records:", missing_record_count)
print(
    "Unsupported/unmatched extracted records:",
    unsupported_record_count
)
print("Schema valid:", schema_validity)
print(
    "Normalisation integrity passed:",
    representation_integrity[
        "normalisation_integrity_passed"
    ]
)
print("Fully correct records:", fully_correct_record_count)
print("Discrepant records:", discrepant_record_count)
print(
    "Overall primary field accuracy:",
    overall_primary_field_accuracy
)
print("Exact record precision:", record_precision_exact)
print("Exact record recall:", record_recall_exact)
print("Exact record F1:", record_f1_exact)

print("\nD6 Validation C completed successfully.")

Validation status: Completed with discrepancies
Reference records: 147
Extracted records: 147
Aligned records: 135
Missing records: 12
Unsupported/unmatched extracted records: 12
Schema valid: True
Normalisation integrity passed: True
Fully correct records: 74
Discrepant records: 61
Overall primary field accuracy: 0.9333333333333333
Exact record precision: 0.5034013605442177
Exact record recall: 0.5034013605442177
Exact record F1: 0.5034013605442177

D6 Validation C completed successfully.


In [ ]:
# ============================================================
# 30. Download generated validation outputs
# ============================================================

GENERATED_OUTPUTS = [
    DETAILED_PATH,
    FULLY_CORRECT_RECORDS_PATH,
    DISCREPANT_RECORDS_PATH,
    MISSING_RECORDS_PATH,
    UNSUPPORTED_RECORDS_PATH,
    SCHEMA_ISSUES_PATH,
    TYPE_ISSUES_PATH,
    FIELD_DISCREPANCIES_PATH,
    FIELD_ACCURACY_PATH,
    CATEGORY_METRICS_PATH,
    VALIDATION_SUMMARY_CSV_PATH,
    VALIDATION_SUMMARY_JSON_PATH,
    VALIDATION_METADATA_PATH,
    VALIDATION_CONCLUSION_PATH
]

print("Generated D6 Validation C files:\n")

for output_path in GENERATED_OUTPUTS:
    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

for output_path in GENERATED_OUTPUTS:
    if output_path.exists():
        files.download(output_path)

Generated D6 Validation C files:

- D6_branch_C_validation_detailed.csv | exists: True
- D6_branch_C_fully_correct_records.csv | exists: True
- D6_branch_C_discrepant_records.csv | exists: True
- D6_branch_C_missing_records.csv | exists: True
- D6_branch_C_unsupported_records.csv | exists: True
- D6_branch_C_schema_issues.csv | exists: True
- D6_branch_C_type_issues.csv | exists: True
- D6_branch_C_field_discrepancies.csv | exists: True
- D6_branch_C_field_error_summary.csv | exists: True
- D6_branch_C_category_metrics.csv | exists: True
- D6_branch_C_validation_summary.csv | exists: True
- D6_branch_C_validation_summary.json | exists: True
- D6_branch_C_validation_metadata.json | exists: True
- D6_branch_C_validation_conclusion.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>